# Predicting Irrigation Need
**`Playground Series - Season 6 Episode 4`**

**GOAL:** Predict the irrigation need.

## Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Import Libraries
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 44

In [3]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold

In [4]:
import gc
import torch
print(torch.cuda.is_available())

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

True


In [5]:
import optuna
from optuna.samplers import TPESampler

## Read Dataset

In [6]:
train_df = pd.read_csv("../Data/train.csv")
test_df = pd.read_csv("../Data/test.csv")

## Preprocessing

In [7]:
def preprocess_and_engineer_features(dataframe):
    # Copy dataframe, so the original data doesn't change.
    dataframe = dataframe.copy()

    # Standardize Column Names
    dataframe.columns = [
        col.upper() for col in dataframe.columns
    ]

    # Map features
    dataframe['MULCHING_USED'] = dataframe['MULCHING_USED'].map(
        {'Yes': 1, 'No': 0}
    )

    # Engineer Features
    dataframe['EVAPORATION_INDEX'] = dataframe['TEMPERATURE_C'] * dataframe['WIND_SPEED_KMH']
    dataframe['MOISTURE_PRESENT'] = dataframe['SOIL_MOISTURE'] * dataframe['RAINFALL_MM']
    dataframe['STRESS_INDEX'] = dataframe['EVAPORATION_INDEX'] / dataframe['SOIL_MOISTURE']
    dataframe['MOISTURE_TEMP_RATIO'] = dataframe['SOIL_MOISTURE'] / dataframe['TEMPERATURE_C']
    dataframe['IS_GROWING_SEASON'] = dataframe['CROP_GROWTH_STAGE'].apply(
        lambda x: 1 if x in ['Flowering', 'Vegetative'] else 0
    )

    # Generate threshold columns
    dataframe['SOIL_MOISTURE_LT_25'] = (dataframe['SOIL_MOISTURE'] < 25).astype(int)
    dataframe['TEMP_GT_30'] = (dataframe['TEMPERATURE_C'] > 30).astype(int)
    dataframe['RAIN_LT_300'] = (dataframe['RAINFALL_MM'] < 300).astype(int)
    dataframe['WIND_GT_10'] = (dataframe['WIND_SPEED_KMH'] > 10).astype(int)

    # Drop near-useless columns
    columns_to_drop = [
        'SOIL_TYPE', 'CROP_TYPE', 'SEASON', 'IRRIGATION_TYPE', 'WATER_SOURCE', 'REGION', 'SOIL_PH',
        'ORGANIC_CARBON', 'ELECTRICAL_CONDUCTIVITY', 'HUMIDITY', 'SUNLIGHT_HOURS', 'FIELD_AREA_HECTARE', 'PREVIOUS_IRRIGATION_MM'
    ]
    dataframe = dataframe.drop(columns = columns_to_drop)
    return dataframe

In [8]:
def build_preprocessor(numerical_features, categorical_features):
    if len(categorical_features) > 0:
        preprocessor = ColumnTransformer(
            transformers = [
                ('num', StandardScaler(), numerical_features),
                ('cat', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore'), categorical_features)
            ]
        )
    else:
        preprocessor = ColumnTransformer(
            transformers = [
                ('num', StandardScaler(), numerical_features)
            ]
        )
    return preprocessor

def fit_preprocessor(train_dataframe, numerical_features, categorical_features):
    preprocessor = build_preprocessor(numerical_features, categorical_features)
    preprocessor.fit(train_dataframe)
    return preprocessor

def transform(dataframe, preprocessor, numerical_features, categorical_features):
    transformed = preprocessor.transform(dataframe)
    if len(categorical_features) > 0:
        feature_names = (
            numerical_features + 
            preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
        )
    else:
        feature_names = numerical_features
    return pd.DataFrame(transformed, columns = feature_names)

## Train-Val Split

In [9]:
train_engineered, test_engineered = preprocess_and_engineer_features(train_df), preprocess_and_engineer_features(test_df)
numerical_features = train_engineered.select_dtypes(include = 'number').columns.tolist()
categorical_features = train_engineered.select_dtypes(include = 'object').columns.tolist()

if 'ID' in numerical_features:
    numerical_features.remove('ID')

if 'IRRIGATION_NEED' in categorical_features:
    categorical_features.remove('IRRIGATION_NEED')

In [10]:
numerical_features, categorical_features

(['SOIL_MOISTURE',
  'TEMPERATURE_C',
  'RAINFALL_MM',
  'WIND_SPEED_KMH',
  'MULCHING_USED',
  'EVAPORATION_INDEX',
  'MOISTURE_PRESENT',
  'STRESS_INDEX',
  'MOISTURE_TEMP_RATIO',
  'IS_GROWING_SEASON',
  'SOIL_MOISTURE_LT_25',
  'TEMP_GT_30',
  'RAIN_LT_300',
  'WIND_GT_10'],
 ['CROP_GROWTH_STAGE'])

In [11]:
X_full, y_full = train_engineered.copy(), train_engineered['IRRIGATION_NEED']
X_full = X_full.drop(columns = ['IRRIGATION_NEED'])
y_full = y_full.map(
    {'Low': 0, 'Medium': 1, 'High': 2}
)

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full,
    test_size = 0.2,
    train_size = 0.8,
    random_state = SEED,
    shuffle = True,
    stratify = y_full
)

In [13]:
preprocessor = fit_preprocessor(X_train, numerical_features, categorical_features)

X_train = transform(X_train, preprocessor, numerical_features, categorical_features)
X_val = transform(X_val, preprocessor, numerical_features, categorical_features)
X_test = transform(test_engineered, preprocessor, numerical_features, categorical_features)

X_train.columns = [
    col.upper() for col in X_train.columns
]

X_val.columns = [
    col.upper() for col in X_val.columns
]

X_test.columns = [
    col.upper() for col in X_test.columns
]

X_full = pd.concat([X_train, X_val], axis = 0).reset_index(drop = True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop = True)

## Baseline Models

In [14]:
weights = compute_class_weight('balanced', classes = np.array([0, 1, 2]), y = y_train)
class_weight_dict = dict(zip([0, 1, 2], weights))
print(class_weight_dict)

{0: np.float64(0.5676941480194908), 1: np.float64(0.8783900365472997), 2: np.float64(9.995835068721366)}


### Logistic Regression

In [15]:
lr_model = Pipeline(
    steps = [
        (
            'model', LogisticRegression(
                multi_class = "multinomial",
                solver = "lbfgs",
                class_weight = class_weight_dict,
                max_iter = 5000,
                random_state = SEED,
                n_jobs = -1
            )
        )
    ]
)

In [16]:
lr_model.fit(X_train, y_train)

D:\Anaconda\envs\kagglePS\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,"{0: np.float64(0.5676941480194908), 1: np.float64(0.8783900365472997), 2: np.float64(9.995835068721366)}"


In [17]:
lr_y_preds = lr_model.predict(X_val)
lr_accuracy = balanced_accuracy_score(y_val, lr_y_preds)
print(f"Accuracy for Logistic Regression: {lr_accuracy:.4f}")

Accuracy for Logistic Regression: 0.9624


### Random Forest

In [18]:
rf_model = Pipeline(
    steps = [
        (
            'model', RandomForestClassifier(
                n_estimators = 100,
                random_state = SEED,
                class_weight = class_weight_dict
            )
        )
    ]
)

In [19]:
rf_model.fit(X_train, y_train)

,steps,"[('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'


In [20]:
rf_y_preds = rf_model.predict(X_val)
rf_accuracy = balanced_accuracy_score(y_val, rf_y_preds)
print(f"Accuracy for Random Forest: {rf_accuracy:.4f}")

Accuracy for Random Forest: 0.9610


## Model - XGBoost

In [21]:
xgb_model = XGBClassifier(
    n_estimators = 100,
    random_state = SEED,
    device = 'cuda',
    eval_metric = 'mlogloss'
)

In [22]:
sample_weights = compute_sample_weight('balanced', y_train)
xgb_model.fit(X_train, y_train, sample_weight = sample_weights)

D:\Anaconda\envs\kagglePS\lib\site-packages\xgboost\training.py:200: UserWarning: [13:55:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [23]:
xgb_y_preds = xgb_model.predict(X_val)
xgb_accuracy = balanced_accuracy_score(y_val, xgb_y_preds)
print(f"Accuracy for XGBoost: {xgb_accuracy:.4f}")

Accuracy for XGBoost: 0.9689


D:\Anaconda\envs\kagglePS\lib\site-packages\xgboost\core.py:751: UserWarning: [13:55:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


## Model - Neural Network

In [24]:
class IrrigationDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype = torch.float32)
        self.y = torch.tensor(y.values, dtype = torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [25]:
class IrrigationNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
    
    def forward(self, x):
        return self.network(x)

In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Datasets and Dataloaders
train_dataset = IrrigationDataset(X_train, y_train)
val_dataset = IrrigationDataset(X_val, y_val)

X_train_tensor = train_dataset.X.to(device)
y_train_tensor = train_dataset.y.to(device)

# Model
input_dim = X_train.shape[1]
model = IrrigationNN(input_dim).to(device)

# Training config
epochs = 100
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.CrossEntropyLoss(weight = torch.tensor(list(class_weight_dict.values()), dtype = torch.float32).to(device))

# Training loop
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

    if epoch == 0 or (epoch+1) % 10 == 0:
        print(f"Epoch: {epoch}, Loss: {loss.item():.4f}")

Device: cuda
Epoch: 0, Loss: 1.1451
Epoch: 9, Loss: 0.5331
Epoch: 19, Loss: 0.2946
Epoch: 29, Loss: 0.1833
Epoch: 39, Loss: 0.1462
Epoch: 49, Loss: 0.1314
Epoch: 59, Loss: 0.1258
Epoch: 69, Loss: 0.1235
Epoch: 79, Loss: 0.1196
Epoch: 89, Loss: 0.1183
Epoch: 99, Loss: 0.1165


In [27]:
model.eval()
with torch.no_grad():
    output = model(val_dataset.X.to(device))
    preds = output.argmax(dim = 1).cpu().numpy()

nn_accuracy = balanced_accuracy_score(y_val, preds)
print(f"Accuracy for Neural Network: {nn_accuracy:.4f}")

Accuracy for Neural Network: 0.9630


## Model Comparisons

In [28]:
model_metrics = {
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Neural Network"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy,
        nn_accuracy
    ]
}

In [29]:
comparison_df = pd.DataFrame(model_metrics)
comparison_df

,Model,Accuracy
0,Logistic Regression,0.962429
1,Random Forest,0.961031
2,XGBoost,0.968939
3,Neural Network,0.963018


## Hyperparameter Tuning

In [30]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 500),
        'max_depth': 3,
        'learning_rate': trial.suggest_float('learning_rate', 0.2, 0.5, log = True),
        'subsample': trial.suggest_float('subsample', 0.85, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.85, 1.0),
        'min_child_weight': 1,
        'device': 'cuda',
        'eval_metric': 'mlogloss',
        'random_state': SEED
    }
    
    skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = SEED)
    scores = []
    
    for train_idx, val_idx in skf.split(X_full, y_full):
        X_tr, X_v = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_v = y_full.iloc[train_idx], y_full.iloc[val_idx]
        
        sample_weights = compute_sample_weight('balanced', y_tr)
        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight = sample_weights)
        preds = model.predict(X_v)
        scores.append(balanced_accuracy_score(y_v, preds))

    return np.mean(scores)

In [31]:
xgb_study = optuna.create_study(direction = 'maximize', sampler = TPESampler(seed = SEED))
xgb_study.optimize(xgb_objective, n_trials = 50, n_jobs = 1)
print("Best params:", xgb_study.best_params)
print("Best accuracy:", xgb_study.best_value)

[I 2026-05-01 13:56:04,528] A new study created in memory with name: no-name-eaf90bd2-6c2b-4505-9167-00ac802ecff1
[I 2026-05-01 13:56:33,402] Trial 0 finished with value: 0.9704357670860716 and parameters: {'n_estimators': 484, 'learning_rate': 0.22015703027599615, 'subsample': 0.9616960722519277, 'colsample_bytree': 0.9040751254384428}. Best is trial 0 with value: 0.9704357670860716.
[I 2026-05-01 13:56:59,558] Trial 1 finished with value: 0.97044035462228 and parameters: {'n_estimators': 436, 'learning_rate': 0.3495186241407463, 'subsample': 0.9090669326632377, 'colsample_bytree': 0.9113608914791348}. Best is trial 1 with value: 0.97044035462228.
[I 2026-05-01 13:57:26,566] Trial 2 finished with value: 0.9701775965029474 and parameters: {'n_estimators': 451, 'learning_rate': 0.3833772115657346, 'subsample': 0.9940789337808501, 'colsample_bytree': 0.9184931663408405}. Best is trial 1 with value: 0.97044035462228.
[I 2026-05-01 13:57:54,028] Trial 3 finished with value: 0.9703996221277

Best params: {'n_estimators': 489, 'learning_rate': 0.24884669985409397, 'subsample': 0.8900126624635736, 'colsample_bytree': 0.9750938767144627}
Best accuracy: 0.9707737334740358


In [32]:
def base_model(trial, input_size):
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    neurons_1 = trial.suggest_categorical("neurons_1", [128, 256])
    neurons_2 = trial.suggest_categorical("neurons_2", [64, 128])
    neurons_3 = trial.suggest_categorical("neurons_3", [32, 64])
    neurons_4 = trial.suggest_categorical("neurons_4", [16, 32])
    
    return nn.Sequential(
        nn.Linear(input_size, neurons_1),
        nn.ReLU(),
        nn.BatchNorm1d(neurons_1),
        nn.Dropout(dropout),
        nn.Linear(neurons_1, neurons_2),
        nn.ReLU(),
        nn.BatchNorm1d(neurons_2),
        nn.Dropout(dropout),
        nn.Linear(neurons_2, neurons_3),
        nn.ReLU(),
        nn.BatchNorm1d(neurons_3),
        nn.Linear(neurons_3, neurons_4),
        nn.ReLU(),
        nn.Linear(neurons_4, 3),
    )

In [33]:
def objective(trial):
    # Generate Model
    model = base_model(trial, input_size).to(device)

    # Optimizer
    optimizer_name = "Adam"
    trial.set_user_attr("optimizer", optimizer_name)
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log = True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log = True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr = lr, weight_decay = weight_decay)
    criterion = nn.CrossEntropyLoss(weight = torch.tensor(list(class_weight_dict.values()), dtype = torch.float32).to(device))

    # Training loop
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_tensor)
        loss = criterion(output, y_train_tensor)
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        output = model(X_val_tensor)
        preds = output.argmax(dim = 1)
    
    accuracy = balanced_accuracy_score(y_val_tensor, preds.cpu().numpy())
    trial.report(accuracy, epoch)
    return accuracy

In [34]:
epochs = 100
input_size = X_train.shape[1]
X_train_tensor = train_dataset.X.to(device)
y_train_tensor = train_dataset.y.to(device)

X_val_tensor = val_dataset.X.to(device)
y_val_tensor = val_dataset.y

nn_study = optuna.create_study(direction = "maximize")
nn_study.optimize(
    objective,
    n_trials = 50,
    n_jobs = 1
)

print("Best params:", nn_study.best_params)
print("Best accuracy:", nn_study.best_value)

[I 2026-05-01 14:19:37,232] A new study created in memory with name: no-name-cbe2d986-51fa-478d-a4c3-9d3368831aac
[I 2026-05-01 14:20:32,162] Trial 0 finished with value: 0.9633040436176561 and parameters: {'dropout': 0.22595092153580035, 'neurons_1': 256, 'neurons_2': 64, 'neurons_3': 32, 'neurons_4': 16, 'lr': 0.005057160077166455, 'weight_decay': 0.00033689772000581935}. Best is trial 0 with value: 0.9633040436176561.
[I 2026-05-01 14:21:27,544] Trial 1 finished with value: 0.9534689863400909 and parameters: {'dropout': 0.16293077602764675, 'neurons_1': 256, 'neurons_2': 64, 'neurons_3': 32, 'neurons_4': 32, 'lr': 0.05650501167886282, 'weight_decay': 0.00041861036056151405}. Best is trial 0 with value: 0.9633040436176561.
[I 2026-05-01 14:22:11,597] Trial 2 finished with value: 0.9275642460040913 and parameters: {'dropout': 0.17350255551772162, 'neurons_1': 128, 'neurons_2': 128, 'neurons_3': 32, 'neurons_4': 32, 'lr': 0.0767284632486894, 'weight_decay': 0.0039733506035199324}. Best

Best params: {'dropout': 0.14117192908494042, 'neurons_1': 256, 'neurons_2': 64, 'neurons_3': 64, 'neurons_4': 16, 'lr': 0.004798985201660906, 'weight_decay': 0.00018430152088558585}
Best accuracy: 0.9640412957048454


In [35]:
torch.cuda.empty_cache()
gc.collect()

0

## Final Model Training

In [36]:
# Best params from Optuna that achieved highest Kaggle score (0.97002)
"""
xgb_best_params = {
    'n_estimators': 452,
    'max_depth': 3,
    'learning_rate': 0.2615597751797821,
    'subsample': 0.9043098095516852,
    'colsample_bytree': 0.9451227797250111,
    'min_child_weight': 1,
    'device': 'cuda',
    'eval_metric': 'mlogloss',
    'random_state': 44
}
xgb_model = XGBClassifier(**xgb_best_params)
"""

xgb_model = XGBClassifier(**xgb_study.best_params)

X_full = pd.concat([X_train, X_val], axis = 0).reset_index(drop = True)
y_full = pd.concat([y_train, y_val], axis = 0).reset_index(drop = True)

sample_weights = compute_sample_weight('balanced', y_full)
xgb_model.fit(X_full, y_full, sample_weight = sample_weights)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9750938767144627
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [37]:
nn_best_params = nn_study.best_params

final_model = nn.Sequential(
    nn.Linear(input_size, nn_best_params['neurons_1']),
    nn.ReLU(),
    nn.BatchNorm1d(nn_best_params['neurons_1']),
    nn.Dropout(nn_best_params['dropout']),
    nn.Linear(nn_best_params['neurons_1'], nn_best_params['neurons_2']),
    nn.ReLU(),
    nn.BatchNorm1d(nn_best_params['neurons_2']),
    nn.Dropout(nn_best_params['dropout']),
    nn.Linear(nn_best_params['neurons_2'], nn_best_params['neurons_3']),
    nn.ReLU(),
    nn.BatchNorm1d(nn_best_params['neurons_3']),
    nn.Linear(nn_best_params['neurons_3'], nn_best_params['neurons_4']),
    nn.ReLU(),
    nn.Linear(nn_best_params['neurons_4'], 3)
).to(device)

optimizer =  getattr(optim, "Adam")(final_model.parameters(), lr = nn_best_params['lr'])
weights = compute_class_weight('balanced', classes = np.array([0, 1, 2]), y = y_full.values)
class_weights = torch.tensor(weights, dtype = torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [38]:
final_model

Sequential(
  (0): Linear(in_features=18, out_features=256, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.14117192908494042, inplace=False)
  (4): Linear(in_features=256, out_features=64, bias=True)
  (5): ReLU()
  (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.14117192908494042, inplace=False)
  (8): Linear(in_features=64, out_features=64, bias=True)
  (9): ReLU()
  (10): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): Linear(in_features=64, out_features=16, bias=True)
  (12): ReLU()
  (13): Linear(in_features=16, out_features=3, bias=True)
)

In [39]:
epochs = 2000
X_full_tensor = torch.cat([X_train_tensor, X_val_tensor.to(device)], dim = 0)
y_full_tensor = torch.cat([y_train_tensor, y_val_tensor.to(device)], dim = 0)

for epoch in range(epochs):
    final_model.train()
    optimizer.zero_grad()
    output = final_model(X_full_tensor)
    loss = criterion(output, y_full_tensor)
    loss.backward()
    optimizer.step()

    if(epoch == 0 or (epoch+1) % 100 == 0):
        print(f"Epoch: {epoch}, Loss: {loss.item():.4f}")

Epoch: 0, Loss: 1.1214
Epoch: 99, Loss: 0.1029
Epoch: 199, Loss: 0.0982
Epoch: 299, Loss: 0.0969
Epoch: 399, Loss: 0.0962
Epoch: 499, Loss: 0.0949
Epoch: 599, Loss: 0.0938
Epoch: 699, Loss: 0.0924
Epoch: 799, Loss: 0.0912
Epoch: 899, Loss: 0.0899
Epoch: 999, Loss: 0.0886
Epoch: 1099, Loss: 0.0876
Epoch: 1199, Loss: 0.0865
Epoch: 1299, Loss: 0.0853
Epoch: 1399, Loss: 0.0844
Epoch: 1499, Loss: 0.0840
Epoch: 1599, Loss: 0.0831
Epoch: 1699, Loss: 0.0816
Epoch: 1799, Loss: 0.0821
Epoch: 1899, Loss: 0.0816
Epoch: 1999, Loss: 0.0804


## Submission Generation

In [40]:
# Test predictions
X_test_array = X_test.values
xgb_preds = xgb_model.predict(X_test_array)

label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
preds_labels = [label_map[p] for p in xgb_preds]

submission = pd.DataFrame({
    'id': test_engineered['ID'],
    'Irrigation_Need': preds_labels
})
submission.to_csv('../Submission/Kaggle_Submission_XGB.csv', index = False)

In [41]:
X_test_tensor = torch.tensor(X_test.values, dtype = torch.float32).to(device)

final_model.eval()
with torch.no_grad():
    output = final_model(X_test_tensor)
    preds = output.argmax(dim = 1).cpu().numpy()

# Map back to original labels
label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
preds_labels = [label_map[p] for p in preds]

# Create submission file
submission = pd.DataFrame({
    'id': test_engineered['ID'],
    'Irrigation_Need': preds_labels
})

submission.to_csv('../Submission/Kaggle_Submission_NN.csv', index = False)